In [ ]:
# TASK 1: CUSTOMER BEHAVIOR ANALYSIS (Alfido Tech)
# Dataset: ecommerce_customer_data_large.csv
# Deliverables:
#   1. Data Cleaning & Preprocessing
#   2. RFM Feature Engineering & K-Means Clustering
#   3. Behavioral Visualizations
#   4. 5 Actionable Recommendations for Alfido Tech

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from google.colab import files

# Set aesthetic visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

In [ ]:

# STEP 1: LOAD & PREPROCESS DATASET
print("Please upload 'ecommerce_customer_data_large.csv':")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("\n--- DATASET OVERVIEW ---")
print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")
print("\nFirst 5 Rows:")
print(df.head())

# Clean column names (strip spaces, lowercase)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Data Cleaning: Handle Missing Values & Duplicates
print("\n--- DATA CLEANING ---")
print("Missing values per column:\n", df.isnull().sum())
print("Duplicate rows count:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

# Fill missing numerical values with median / categorical with mode if present
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

Please upload 'ecommerce_customer_data_large.csv':


In [ ]:

# --- 1. Price & Transaction Features ---
if 'total_purchase_amount' in df.columns and 'quantity' in df.columns:
    # Calculate price per individual unit
    df['price_per_unit'] = df['total_purchase_amount'] / df['quantity']

    # Flag high-value transactions (above average total spend)
    mean_spend = df['total_purchase_amount'].mean()
    df['is_high_value_purchase'] = (df['total_purchase_amount'] > mean_spend).astype(int)

# --- 2. Customer Aggregation Features ---
if 'customer_id' in df.columns and 'total_purchase_amount' in df.columns:
    # Total historical spend per customer
    df['customer_total_spend'] = df.groupby('customer_id')['total_purchase_amount'].transform('sum')

    # Total purchase frequency per customer
    df['purchase_frequency'] = df.groupby('customer_id')['customer_id'].transform('count')

    # Average order value per customer
    df['avg_order_value'] = df['customer_total_spend'] / df['purchase_frequency']

# --- 3. Categorical Binning ---
if 'age' in df.columns:
    # Segment age into demographic groups
    df['age_group'] = pd.cut(
        df['age'],
        bins=[0, 25, 40, 60, 100],
        labels=['Young', 'Adult', 'Middle_Aged', 'Senior']
    )

if 'total_purchase_amount' in df.columns:
    # Quantile-based spending category
    df['spending_tier'] = pd.qcut(
        df['total_purchase_amount'],
        q=3,
        labels=['Low', 'Medium', 'High']
    )

# --- 4. Return Behavior Flag ---
if 'returns' in df.columns:
    df['has_returned'] = (df['returns'] > 0).astype(int)

In [ ]:
# Display summary of engineered features
print("\nCleaned & Engineered Dataset Info:")
print(df.info())

print("\nFirst 5 records:")
print(df.head())

# Save cleaned and engineered dataset
df.to_csv('cleaned_ecommerce_dataset.csv', index=False)

In [ ]:
# 2. FEATURE MAPPING FOR RFM ANALYSIS
# ------------------------------------------------------------------------------
# Helper for dynamic column identification
def detect_col(keywords, df_cols):
    for kw in keywords:
        matches = [c for c in df_cols if kw in c]
        if matches:
            return matches[0]
    return None

recency_col = detect_col(['recency', 'days_since_last', 'last_purchase'], df.columns)
frequency_col = detect_col(['total_orders', 'total_purchases', 'frequency', 'purchases'], df.columns)

# If monetary isn't directly present, calculate total spend = total_purchases * avg_purchase_value
if detect_col(['total_spend', 'monetary', 'annual_income'], df.columns):
    monetary_col = detect_col(['total_spend', 'monetary'], df.columns)
    df['Monetary'] = df[monetary_col]
elif 'avg_purchase_value' in df.columns and frequency_col in df.columns:
    df['Monetary'] = df['avg_purchase_value'] * df[frequency_col]
else:
    df['Monetary'] = df[detect_col(['spend', 'value', 'amount'], df.columns)]


df['Frequency'] = df[frequency_col]

In [ ]:
# 2. DERIVE KEY VISUALIZATION METRICS
# ------------------------------------------------------------------------------
# Dynamic column identification

freq_col = [c for c in df.columns if 'purchase' in c or 'order' in c or 'freq' in c][0]
spend_col = [c for c in df.columns if 'spend' in c or 'total' in c or 'amount' in c or 'price' in c][0]

df['Purchase_Frequency'] = df[freq_col]
df['Total_Spend'] = df[spend_col]

# Define retention risk cohorts based on recency
bins = [-1, 30, 60, 90, 365]
labels = ['Active (<30 days)', 'Slight Risk (31-60 days)', 'High Risk (61-90 days)', 'Lapsed (>90 days)']




In [ ]:
# 3. CREATE 4-PANEL PURCHASE & RETENTION DASHBOARD
# ------------------------------------------------------------------------------

# Re-calculate customer-level metrics required for the dashboard within this cell
# to ensure 'Retention_Cohort' and other aggregated metrics are available.

# Ensure purchase_date is datetime, if not already
if not pd.api.types.is_datetime64_any_dtype(df['purchase_date']):
    df['purchase_date'] = pd.to_datetime(df['purchase_date'])

# Calculate customer-level aggregated metrics
customer_metrics_df = df.groupby('customer_id').agg(
    # Recency: days since last purchase
    Recency_Days=('purchase_date', lambda x: (df['purchase_date'].max() - x.max()).days),
    # Purchase Frequency: count of unique purchases (each row in df is a purchase)
    Purchase_Frequency=('customer_id', 'count'),
    # Total Spend: sum of all purchase amounts
    Total_Spend=('total_purchase_amount', 'sum'),
    # Average Order Value (AOV): total spend / number of purchases
    AOV=('total_purchase_amount', lambda x: x.sum() / x.count())
).reset_index()

# Define retention risk cohorts based on recency
bins = [-1, 30, 60, 90, 365]
labels = ['Active (<30 days)', 'Slight Risk (31-60 days)', 'High Risk (61-90 days)', 'Lapsed (>90 days)']
customer_metrics_df['Retention_Cohort'] = pd.cut(
    customer_metrics_df['Recency_Days'],
    bins=bins,
    labels=labels,
    right=True
)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: Customer Retention Status Distribution
retention_counts = customer_metrics_df['Retention_Cohort'].value_counts().sort_index()
sns.barplot(
    x=retention_counts.index, y=retention_counts.values,
    palette='Blues_d', ax=axes[0, 0]
)
axes[0, 0].set_title('1. Customer Retention Status Distribution', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Number of Customers')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=15)

# Annotate values on top of bars
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 5),
                        textcoords='offset points')

# Chart 2: Purchase Frequency vs Total Spend Heatmap/Scatter Pattern
sns.scatterplot(
    data=customer_metrics_df, x='Purchase_Frequency', y='Total_Spend',
    hue='Retention_Cohort', palette='Spectral', alpha=0.8, s=60, ax=axes[0, 1]
)
axes[0, 1].set_title('2. Purchase Pattern: Frequency vs Total Spend', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Total Order Frequency')
axes[0, 1].set_ylabel('Total Spend ($)')

# Chart 3: Retention Risk vs Average Order Value (AOV)
sns.boxplot(
    data=customer_metrics_df, x='Retention_Cohort', y='AOV',
    palette='Set2', ax=axes[1, 0]
)
axes[1, 0].set_title('3. Average Order Value (AOV) across Retention Cohorts', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Average Order Value ($)')
axes[1, 0].set_xlabel('')
axes[1, 0].tick_params(axis='x', rotation=15)

# Chart 4: Cumulative Monetary Contribution by Active vs Risk Groups
retention_spend = customer_metrics_df.groupby('Retention_Cohort')['Total_Spend'].sum().reset_index()
axes[1, 1].pie(
    retention_spend['Total_Spend'],
    labels=retention_spend['Retention_Cohort'],
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('Set3', 4)
)
axes[1, 1].set_title('4. Total Revenue Share by Retention Cohort', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('purchase_patterns_and_retention_trends.png', dpi=300)
plt.show()

print("\n Visualization dashboard generated and saved as 'purchase_patterns_and_retention_trends.png'.")